In [1]:
import boto3
import pandas as pd
from io import BytesIO
import duckdb
import os

# --- AWS LocalStack Config ---
s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:4566",  # LocalStack endpoint
    aws_access_key_id="test",
    aws_secret_access_key="test",
    region_name="us-east-1"
)

bucket_name = "zazi-data-bucket"

# --- 1️⃣ Step 1: Create sample data ---
data = {
    "customer_id": [101, 102, 103, 104],
    "name": ["Zazi", "Amahle", "Kwena", "Nathi"],
    "city": ["Cape Town", "Joburg", "Durban", "Pretoria"],
    "purchase_value": [250, 400, 320, 180]
}

df = pd.DataFrame(data)
print("✅ Original Data")
display(df)

# --- 2️⃣ Step 2: Transform data ---
df["purchase_value_with_vat"] = df["purchase_value"] * 1.15
df["segment"] = df["purchase_value"].apply(lambda x: "High" if x > 300 else "Low")

print("✅ Transformed Data")
display(df)

# --- 3️⃣ Step 3: Save as Parquet ---
output_file = "zazi_etl_output.parquet"
df.to_parquet(output_file, index=False)

print(f"✅ Parquet file created: {output_file}")

# --- 4️⃣ Step 4: Upload to LocalStack S3 ---
s3.upload_file(output_file, bucket_name, f"etl-output/{output_file}")
print("✅ Uploaded to Local S3 successfully!")

# --- 5️⃣ Step 5: Validate upload ---
response = s3.list_objects_v2(Bucket=bucket_name)
print("✅ Files in S3:")
for obj in response.get("Contents", []):
    print("-", obj["Key"])

# --- 6️⃣ Step 6: Query with DuckDB (like Athena) ---
con = duckdb.connect()
df_query = con.execute("""
    SELECT city, SUM(purchase_value_with_vat) as total_spent
    FROM read_parquet('zazi_etl_output.parquet')
    GROUP BY city
    ORDER BY total_spent DESC
""").fetchdf()

print("✅ Aggregated Data (Athena-style Query):")
display(df_query)


✅ Original Data


,customer_id,name,city,purchase_value
0,101,Zazi,Cape Town,250
1,102,Amahle,Joburg,400
2,103,Kwena,Durban,320
3,104,Nathi,Pretoria,180


✅ Transformed Data


,customer_id,name,city,purchase_value,purchase_value_with_vat,segment
0,101,Zazi,Cape Town,250,287.5,Low
1,102,Amahle,Joburg,400,460.0,High
2,103,Kwena,Durban,320,368.0,High
3,104,Nathi,Pretoria,180,207.0,Low


✅ Parquet file created: zazi_etl_output.parquet
✅ Uploaded to Local S3 successfully!
✅ Files in S3:
- etl-output/zazi_etl_output.parquet
✅ Aggregated Data (Athena-style Query):


,city,total_spent
0,Joburg,460.0
1,Durban,368.0
2,Cape Town,287.5
3,Pretoria,207.0


In [2]:
pip install pyarrow --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
docker ps

SyntaxError: invalid syntax (2275867613.py, line 1)

In [4]:
python producer.py

SyntaxError: invalid syntax (1674376850.py, line 1)